In [2]:
d = {1: "Python", 2: "C++"}
print(d[True], d[1.0], d[1])
print(len(d))
# print(d[3])

Python Python Python
2


Why 1, True, and 1.0 Overwrite Each Other in Python Dictionaries

The Explanation:In Python, dictionaries do not care about the data type of a key (whether it is an int, bool, or float). When you assign or look up a key, Python uses a strict two-step process under the hood: Hash and Equality.Step 1: The Hash Check (hash())To find out where to store the data in memory, Python calculates the hash value of the key.Because bool is a subclass of int in Python, True is mathematically treated as 1. Therefore, they all generate the exact same hash:hash(1) $\rightarrow$ 1hash(True) $\rightarrow$ 1hash(1.0) $\rightarrow$ 1Python sees the hash 1 for all three and directs them to the exact same memory bucket.Step 2: The Equality Check (==)Once inside that bucket, Python checks if the new key is mathematically equal to the existing key to decide if it should overwrite the value.1 == True $\rightarrow$ TrueTrue == 1.0 $\rightarrow$ TrueConclusion:Because the hashes match AND the values are mathematically equal, Python's engine concludes that 1, True, and 1.0 are the exact same key. Instead of creating three separate entries, Python keeps the original key layout (1) but overwrites the value in that slot each time.By the end of the script, "C++" is the final value saved, and the dictionary only contains a total of 1 item.

why? Dont we  say thet immutable  can be added in the doct as keys they all are immutables why they are type casting? what if i dont want that? Why python using its brain unnecessary

Python is not actually type casting them behind your back. It is just blindly following the two strict rules of how dictionaries look up keys.Dictionaries do not care about your object's type (whether it is an int, float, or bool). They only care about two things: Hash and Equality.Here is exactly how Python's engine processes data[1], data[1.0], and data[True].The 2-Step Key CheckWhen you try to add or look up a key, Python performs this sequence:Step 1: The Hash CheckPython calculates the hash() of the key to find the correct drawer in the dictionary's memory.Because Python enforces mathematical equivalence:hash(1) $\rightarrow$ 1hash(1.0) $\rightarrow$ 1hash(True) $\rightarrow$ 1 (Because bool is literally a subclass of int in Python, True is physically just the integer 1 under the hood).Python sees the hash 1 for all three, so it puts them all into the exact same memory drawer.Step 2: The Equality Check (==)If Python finds an existing key in that drawer, it checks if the new key is mathematically equal to the old key to see if it should overwrite it.Does 1 == 1.0? True.Does 1.0 == True? True.Because the hashes match AND the values are equal, Python's engine concludes: "These are the exact same key. I must overwrite the old value."Why Python Designed It This Way (Not Unnecessary)It feels annoying here, but Python does this to protect your math logic.Imagine you are calculating a user's ID.If user_id = 5 / 2 * 2, the result is the float 5.0.If your dictionary has the integer 5 as a key, you want data[user_id] to successfully find the user!If Python strictly separated 5 and 5.0 into different dictionary keys just because one is a float and one is an int, it would cause massive, hidden bugs anytime a division operation accidentally converted your integers into floats.By forcing 1, 1.0, and True to act identically in dictionaries, Python guarantees that mathematically equal numbers always point to the same data, regardless of their data type.

so in same manner False ans 0 will be considered one?

Yes, exactly! You have completely mastered the logic.

False, 0, and 0.0 will all overwrite each other in the exact same dictionary slot, for the exact same reasons.

Here is the proof:

In [3]:
data = {}
data[0] = "Zero"
data[False] = "Boolean False"
data[0.0] = "Float Zero"

print(data[0])  # Prints: Float Zero
print(len(data))  # Prints: 1

Float Zero
1


Why it works the exact same way:
Because bool is a subclass of int, False is literally just the integer 0 under the hood.

The Hash Match: hash(0), hash(False), and hash(0.0) all return exactly 0. Python opens drawer #0.

The Equality Match: 0 == False == 0.0 evaluates to True. Python assumes they are identical keys.

You can safely assume that any time numbers are mathematically equal in Python, they will collide in a dictionary or a set.


In [ ]:
# lets try same logc with sets

In [5]:
s = set()
s.add(0)
s.add(False)
s.add(0.0)
s

{0}

In [6]:
len(s)

1

Sets and dictionaries in Python are practically siblings under the hood. A set is essentially just a dictionary that only has keys and no values.

Because they use the exact same C-level hash map architecture, sets enforce the exact same Hash + Equality rule.

Here is the proof of how it plays out in a set:

In [ ]:
my_set = {1.0, 1, True}

print(my_set)  # Output: {1.0}
print(len(my_set))  # Output: 1

{1.0}
1


The "First Come, First Served" Rule
Notice how the output is {1.0} and not {True} or {1}?

With dictionaries, you are actively re-assigning values to a key, so the value constantly gets overwritten. But sets don't have values. They only care about existence.

So, when Python evaluates {1.0, 1, True}, it processes them left to right:

Sees 1.0: Hashes to 1. The drawer is empty. It puts 1.0 inside.

Sees 1: Hashes to 1. Opens the drawer, finds 1.0. Checks 1 == 1.0 (True). Python says, "I already have this exact mathematical value. I will ignore the new one."

Sees True: Hashes to 1. Opens the drawer, finds 1.0. Checks True == 1.0 (True). Python says, "Already have it. Ignore."

The set locks in the very first object that claimed that hash slot, and silently throws away the rest as duplicates.

If you changed the order to my_set = {True, 1.0, 1}, the output would be {True}.

In [1]:
my_set = {True, 1, True, 1.0}

print(my_set)  # Output: {1.0}
print(len(my_set))  # Output: 1

{True}
1


In [ ]:
class StrictKey:
    def __init__(self, value):
        self.value = value

    def __hash__(self):
        # Hash based on BOTH type and value
        return hash((type(self.value), self.value))

    def __eq__(self, other):
        # Force exact type match AND exact value match
        if type(self.value) is not type(other.value):
            return False
        return self.value == other.value


# Now they will never collide
data = {StrictKey(1): "Integer", StrictKey(1.0): "Float", StrictKey(True): "Boolean"}

print(len(data))  # Output: 3

3


In [5]:
data[StrictKey(1.0)]  # Output: "Integer"

'Float'

By wrapping your keys, you take complete control away from Python's default math engine and enforce strict type-checking yourself.

In [10]:
data = {1: "Integer", 1.0: "Float", True: "Boolean"}
print(data[1.0])
print(len(data))  # Output: 1

Boolean
1
